In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/topic/openings-expansions/?page=3"
base_url = "https://www.manufacturingdive.com"

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Find all h3 elements with that class
articles = soup.find_all("h3", class_="feed__title feed__title--display qqq")

urls = []
for article in articles:
    a_tag = article.find("a")
    if a_tag and "href" in a_tag.attrs:
        link = a_tag["href"]
        # Convert relative link to absolute
        if link.startswith("/"):
            link = base_url + link
        urls.append(link)

print(urls)

/Users/mac/Documents/Tuone/tuone_v6/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


['https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/', 'https://www.manufacturingdive.com/news/siemens-acquires-altair-engineering-10-billion-boost-ai-capabilities/743798/', 'https://www.manufacturingdive.com/news/schneider-electric-700-million-investment-us-data-center-demand-ai/743630/', 'https://www.manufacturingdive.com/news/oci-holdings-mission-solar-energy-solar-cell-expansion-san-antonio-texas/743484/', 'https://www.manufacturingdive.com/news/hyundai-investment-us-tariffs-trump-steel-louisiana/743377/', 'https://www.manufacturingdive.com/news/CSW-Industrials-acquire-313-million-hvac-manufacturer-aspen-manufacturing/743383/', 'https://www.manufacturingdive.com/news/jj-boosts-us-manufacturing-55-billion/743310/', 'https://www.manufacturingdive.com/news/t1-energy-850m-solar-cell-facility-rockdale-texas-g2-austin-us-supply-chain/742986/', 'https://www.manufacturingdive.com/news/india-syngene-acquires-first-us-

In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# --- Extract Title ---
title = soup.find("h1").get_text(strip=True)

# --- Extract Paragraphs (inside article body only) ---
paragraphs_dict = {
    f"p{idx + 1}": p.get_text(" ", strip=True)   # keep text clean with spaces
    for idx, p in enumerate(soup.select("div.article__body p"))
}
paragraphs = [paragraphs_dict]

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag["datetime"] if date_tag and "datetime" in date_tag.attrs else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category ---
category_tag = soup.select_one("a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link")
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "paragraphs": paragraphs,
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}

print(article_data)

{'title': "Don't miss tomorrow's Manufacturing industry news", 'paragraphs': [{}], 'meta': {'date': None, 'url': 'https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/', 'category': 'Unknown', 'author': 'Unknown'}}


In [5]:
import requests
from bs4 import BeautifulSoup

def extract_dive_insight(url):
    """
    Extract the Dive Insight section from a Manufacturing Dive article
    """
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # --- Extract Title ---
    title = soup.find("h1")
    title_text = title.get_text(strip=True) if title else "No title found"
    
    # --- Extract Dive Insight Section ---
    # Look for the Dive Insight heading and extract all paragraphs that follow
    dive_insight_paragraphs = []
    dive_insight_section = soup.find("h3", string="Dive Insight:")
    
    if dive_insight_section:
        # Find the parent div that contains the Dive Insight section
        dive_insight_div = dive_insight_section.find_parent("div")
        if dive_insight_div:
            # Extract all paragraphs within this div
            paragraphs = dive_insight_div.find_all("p")
            dive_insight_paragraphs = [p.get_text(" ", strip=True) for p in paragraphs]
    
    # --- Extract Regular Article Paragraphs (inside article body only) ---
    article_paragraphs = []
    article_body = soup.select("div.article__body p")
    if article_body:
        article_paragraphs = [p.get_text(" ", strip=True) for p in article_body]
    
    # --- Extract Date ---
    date_tag = soup.find("time")
    date_utc = date_tag["datetime"] if date_tag and "datetime" in date_tag.attrs else None
    
    # --- Extract Author ---
    author_tag = soup.select_one("a.article-byline__author-link")
    author = author_tag.get_text(strip=True) if author_tag else "Unknown"
    
    # --- Extract Category ---
    category_tag = soup.select_one("a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link")
    category = category_tag.get_text(strip=True) if category_tag else "Unknown"
    
    # --- Build Final Dict ---
    article_data = {
        "title": title_text,
        "dive_insight": dive_insight_paragraphs,
        "article_paragraphs": article_paragraphs,
        "meta": {
            "date": date_utc,
            "url": url,
            "category": category,
            "author": author,
        },
    }
    
    return article_data

def print_article_data(article_data):
    """Pretty print the extracted article data"""
    print("=" * 80)
    print(f"TITLE: {article_data['title']}")
    print("=" * 80)
    
    print(f"\n📅 Date: {article_data['meta']['date']}")
    print(f"👤 Author: {article_data['meta']['author']}")
    print(f"📂 Category: {article_data['meta']['category']}")
    print(f"🔗 URL: {article_data['meta']['url']}")
    
    if article_data['dive_insight']:
        print(f"\n🔍 DIVE INSIGHT SECTION ({len(article_data['dive_insight'])} paragraphs):")
        print("-" * 50)
        for i, p in enumerate(article_data['dive_insight'], 1):
            print(f"\nParagraph {i}:")
            print(p)
            print("-" * 30)
    else:
        print("\n❌ No Dive Insight section found")
    
    if article_data['article_paragraphs']:
        print(f"\n📰 ARTICLE BODY ({len(article_data['article_paragraphs'])} paragraphs):")
        print("-" * 50)
        for i, p in enumerate(article_data['article_paragraphs'][:3], 1):  # Show first 3 paragraphs
            print(f"\nParagraph {i}:")
            print(p[:200] + "..." if len(p) > 200 else p)
            print("-" * 30)
        if len(article_data['article_paragraphs']) > 3:
            print(f"... and {len(article_data['article_paragraphs']) - 3} more paragraphs")

if __name__ == "__main__":
    # Test with the URL you provided
    test_url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"
    
    print("🔍 Extracting Dive Insight from Manufacturing Dive article...")
    print(f"URL: {test_url}\n")
    
    try:
        article_data = extract_dive_insight(test_url)
        print_article_data(article_data)
    except Exception as e:
        print(f"❌ Error extracting data: {e}")
        import traceback
        traceback.print_exc()


🔍 Extracting Dive Insight from Manufacturing Dive article...
URL: https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/

TITLE: Don't miss tomorrow's Manufacturing industry news

📅 Date: None
👤 Author: Unknown
📂 Category: Unknown
🔗 URL: https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/

🔍 DIVE INSIGHT SECTION (10 paragraphs):
--------------------------------------------------

Paragraph 1:
Isopropyl alcohol was the first chemical ExxonMobil produced over 100 years ago, according to the company. It is a byproduct of propylene processing , commonly found in hand sanitizers, household cleaning products, perfumes and lotions.
------------------------------

Paragraph 2:
In 1992, Exxon Mobil started increasing the purity of its cleaning solution for industrial uses, including for microchips, medical devices, laptops and gaming consoles.
----------

In [6]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"

# Add a user-agent + timeout for reliability
resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# --- Extract Title ---
title_tag = soup.find("h1")
title = title_tag.get_text(strip=True) if title_tag else "No title found"

# --- Extract Dive Insight Section (this is what you want!) ---
dive_insight_paragraphs = []
dive_insight_section = soup.find("h3", string="Dive Insight:")

if dive_insight_section:
    # Find the parent div that contains the Dive Insight section
    dive_insight_div = dive_insight_section.find_parent("div")
    if dive_insight_div:
        # Extract all paragraphs within this div
        dive_insight_nodes = dive_insight_div.find_all("p")
        dive_insight_paragraphs = [
            p.get_text(" ", strip=True)
            for p in dive_insight_nodes
            if p.get_text(strip=True)
        ]

# --- Extract Regular Article Paragraphs (fallback) ---
# Try multiple selectors since the structure might vary
article_paragraphs = []
selectors_to_try = [
    "div.article__body p",
    "div.article-content p", 
    "div.content p",
    "article p",
    "div[class*='article'] p",
    "div[class*='content'] p"
]

for selector in selectors_to_try:
    paragraph_nodes = soup.select(selector)
    if paragraph_nodes:
        article_paragraphs = [
            p.get_text(" ", strip=True)
            for p in paragraph_nodes
            if p.get_text(strip=True)
        ]
        print(f"✅ Found {len(article_paragraphs)} paragraphs using selector: {selector}")
        break

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag.get("datetime") if date_tag else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category (fallbacks included) ---
category_tag = soup.select_one(
    "a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link"
)
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "dive_insight": dive_insight_paragraphs,  # <-- This is the key section you want!
    "article_paragraphs": article_paragraphs,  # <-- Regular article content
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}

# --- Print Results ---
print("\n" + "="*80)
print(f"TITLE: {article_data['title']}")
print("="*80)

print(f"\n📅 Date: {article_data['meta']['date']}")
print(f"👤 Author: {article_data['meta']['author']}")
print(f"�� Category: {article_data['meta']['category']}")

if article_data['dive_insight']:
    print(f"\n🔍 DIVE INSIGHT SECTION ({len(article_data['dive_insight'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['dive_insight'], 1):
        print(f"\nParagraph {i}:")
        print(p)
        print("-" * 30)
else:
    print("\n❌ No Dive Insight section found")

if article_data['article_paragraphs']:
    print(f"\n�� ARTICLE BODY ({len(article_data['article_paragraphs'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['article_paragraphs'][:3], 1):  # Show first 3
        print(f"\nParagraph {i}:")
        print(p[:200] + "..." if len(p) > 200 else p)
        print("-" * 30)
    if len(article_data['article_paragraphs']) > 3:
        print(f"... and {len(article_data['article_paragraphs']) - 3} more paragraphs")
else:
    print("\n❌ No regular article paragraphs found")

print("\n" + "="*80)

✅ Found 11 paragraphs using selector: article p

TITLE: Don't miss tomorrow's Manufacturing industry news

📅 Date: None
👤 Author: Unknown
�� Category: Unknown

🔍 DIVE INSIGHT SECTION (10 paragraphs):
--------------------------------------------------

Paragraph 1:
Isopropyl alcohol was the first chemical ExxonMobil produced over 100 years ago, according to the company. It is a byproduct of propylene processing , commonly found in hand sanitizers, household cleaning products, perfumes and lotions.
------------------------------

Paragraph 2:
In 1992, Exxon Mobil started increasing the purity of its cleaning solution for industrial uses, including for microchips, medical devices, laptops and gaming consoles.
------------------------------

Paragraph 3:
As microchips become even smaller, the company said it will need “ultra-pure” isopropyl alcohol to dry wafer surfaces, reduce impurities and prevent damage of the chips’ sensitive circuitry.
------------------------------

Paragraph 4:
To 

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"

# Add a user-agent + timeout for reliability
resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# --- Extract Title ---
title_tag = soup.find("h1")
title = title_tag.get_text(strip=True) if title_tag else "No title found"

# --- Extract Paragraphs (inside article body only) ---
# Keep as an ordered list of strings
paragraph_nodes = soup.select("div.article__body p")
paragraphs = [
    p.get_text(" ", strip=True)
    for p in paragraph_nodes
    if p.get_text(strip=True)
]

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag.get("datetime") if date_tag else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category (fallbacks included) ---
category_tag = soup.select_one(
    "a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link"
)
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "paragraphs": paragraphs,   # <-- now a clean list of paragraph strings
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}
print(article_data)


{'title': "Don't miss tomorrow's Manufacturing industry news", 'paragraphs': [], 'meta': {'date': None, 'url': 'https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/', 'category': 'Unknown', 'author': 'Unknown'}}


In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"

# Add a user-agent + timeout for reliability
resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# --- Extract Title ---
title_tag = soup.find("h1")
title = title_tag.get_text(strip=True) if title_tag else "No title found"

# --- Extract Paragraphs (inside article body only) ---
# Keep as an ordered list of strings
paragraph_nodes = soup.select("div.article__body p")
paragraphs = [
    p.get_text(" ", strip=True)
    for p in paragraph_nodes
    if p.get_text(strip=True)
]

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag.get("datetime") if date_tag else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category (fallbacks included) ---
category_tag = soup.select_one(
    "a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link"
)
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "paragraphs": paragraphs,   # <-- now a clean list of paragraph strings
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}
print(article_data)


{'title': "Don't miss tomorrow's Manufacturing industry news", 'paragraphs': [], 'meta': {'date': None, 'url': 'https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/', 'category': 'Unknown', 'author': 'Unknown'}}


In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"

# Add a user-agent + timeout for reliability
resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# --- Extract Title ---
title_tag = soup.find("h1")
title = title_tag.get_text(strip=True) if title_tag else "No title found"

# --- Extract Paragraphs (inside article body only) ---
# Keep as an ordered list of strings
paragraph_nodes = soup.select("div.article__body p")
paragraphs = [
    p.get_text(" ", strip=True)
    for p in paragraph_nodes
    if p.get_text(strip=True)
]

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag.get("datetime") if date_tag else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category (fallbacks included) ---
category_tag = soup.select_one(
    "a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link"
)
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "paragraphs": paragraphs,   # <-- now a clean list of paragraph strings
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}
print(article_data)


{'title': "Don't miss tomorrow's Manufacturing industry news", 'paragraphs': [], 'meta': {'date': None, 'url': 'https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/', 'category': 'Unknown', 'author': 'Unknown'}}


In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"

# Add a user-agent + timeout for reliability
resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# --- Extract Title ---
title_tag = soup.find("h1")
title = title_tag.get_text(strip=True) if title_tag else "No title found"

# --- Extract Dive Insight Section (this is what you want!) ---
dive_insight_paragraphs = []
dive_insight_section = soup.find("h3", string="Dive Insight:")

if dive_insight_section:
    # Find the parent div that contains the Dive Insight section
    dive_insight_div = dive_insight_section.find_parent("div")
    if dive_insight_div:
        # Extract all paragraphs within this div
        dive_insight_nodes = dive_insight_div.find_all("p")
        dive_insight_paragraphs = [
            p.get_text(" ", strip=True)
            for p in dive_insight_nodes
            if p.get_text(strip=True)
        ]

# --- Extract Regular Article Paragraphs (fallback) ---
# Try multiple selectors since the structure might vary
article_paragraphs = []
selectors_to_try = [
    "div.article__body p",
    "div.article-content p", 
    "div.content p",
    "article p",
    "div[class*='article'] p",
    "div[class*='content'] p"
]

for selector in selectors_to_try:
    paragraph_nodes = soup.select(selector)
    if paragraph_nodes:
        article_paragraphs = [
            p.get_text(" ", strip=True)
            for p in paragraph_nodes
            if p.get_text(strip=True)
        ]
        print(f"✅ Found {len(article_paragraphs)} paragraphs using selector: {selector}")
        break

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag.get("datetime") if date_tag else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category (fallbacks included) ---
category_tag = soup.select_one(
    "a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link"
)
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "dive_insight": dive_insight_paragraphs,  # <-- This is the key section you want!
    "article_paragraphs": article_paragraphs,  # <-- Regular article content
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}

# --- Print Results ---
print("\n" + "="*80)
print(f"TITLE: {article_data['title']}")
print("="*80)

print(f"\n📅 Date: {article_data['meta']['date']}")
print(f"👤 Author: {article_data['meta']['author']}")
print(f"�� Category: {article_data['meta']['category']}")

if article_data['dive_insight']:
    print(f"\n🔍 DIVE INSIGHT SECTION ({len(article_data['dive_insight'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['dive_insight'], 1):
        print(f"\nParagraph {i}:")
        print(p)
        print("-" * 30)
else:
    print("\n❌ No Dive Insight section found")

if article_data['article_paragraphs']:
    print(f"\n�� ARTICLE BODY ({len(article_data['article_paragraphs'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['article_paragraphs'][:3], 1):  # Show first 3
        print(f"\nParagraph {i}:")
        print(p[:200] + "..." if len(p) > 200 else p)
        print("-" * 30)
    if len(article_data['article_paragraphs']) > 3:
        print(f"... and {len(article_data['article_paragraphs']) - 3} more paragraphs")
else:
    print("\n❌ No regular article paragraphs found")

print("\n" + "="*80)

✅ Found 11 paragraphs using selector: article p

TITLE: Don't miss tomorrow's Manufacturing industry news

📅 Date: None
👤 Author: Unknown
�� Category: Unknown

🔍 DIVE INSIGHT SECTION (10 paragraphs):
--------------------------------------------------

Paragraph 1:
Isopropyl alcohol was the first chemical ExxonMobil produced over 100 years ago, according to the company. It is a byproduct of propylene processing , commonly found in hand sanitizers, household cleaning products, perfumes and lotions.
------------------------------

Paragraph 2:
In 1992, Exxon Mobil started increasing the purity of its cleaning solution for industrial uses, including for microchips, medical devices, laptops and gaming consoles.
------------------------------

Paragraph 3:
As microchips become even smaller, the company said it will need “ultra-pure” isopropyl alcohol to dry wafer surfaces, reduce impurities and prevent damage of the chips’ sensitive circuitry.
------------------------------

Paragraph 4:
To 

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"

# Add a user-agent + timeout for reliability
resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# --- Extract Title ---
title_tag = soup.find("h1")
title = title_tag.get_text(strip=True) if title_tag else "No title found"

# --- Extract Dive Insight Section (this is what you want!) ---
dive_insight_paragraphs = []
dive_insight_section = soup.find("h3", string="Dive Insight:")

if dive_insight_section:
    # Find the parent div that contains the Dive Insight section
    dive_insight_div = dive_insight_section.find_parent("div")
    if dive_insight_div:
        # Extract all paragraphs within this div
        dive_insight_nodes = dive_insight_div.find_all("p")
        dive_insight_paragraphs = [
            p.get_text(" ", strip=True)
            for p in dive_insight_nodes
            if p.get_text(strip=True)
        ]

# --- Extract Regular Article Paragraphs (fallback) ---
# Try multiple selectors since the structure might vary
article_paragraphs = []
selectors_to_try = [
    "div.article__body p",
    "div.article-content p", 
    "div.content p",
    "article p",
    "div[class*='article'] p",
    "div[class*='content'] p"
]

for selector in selectors_to_try:
    paragraph_nodes = soup.select(selector)
    if paragraph_nodes:
        article_paragraphs = [
            p.get_text(" ", strip=True)
            for p in paragraph_nodes
            if p.get_text(strip=True)
        ]
        print(f"✅ Found {len(article_paragraphs)} paragraphs using selector: {selector}")
        break

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag.get("datetime") if date_tag else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category (fallbacks included) ---
category_tag = soup.select_one(
    "a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link"
)
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "dive_insight": dive_insight_paragraphs,  # <-- This is the key section you want!
    "article_paragraphs": article_paragraphs,  # <-- Regular article content
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}

# --- Print Results ---
print("\n" + "="*80)
print(f"TITLE: {article_data['title']}")
print("="*80)

print(f"\n📅 Date: {article_data['meta']['date']}")
print(f"👤 Author: {article_data['meta']['author']}")
print(f"�� Category: {article_data['meta']['category']}")

if article_data['dive_insight']:
    print(f"\n🔍 DIVE INSIGHT SECTION ({len(article_data['dive_insight'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['dive_insight'], 1):
        print(f"\nParagraph {i}:")
        print(p)
        print("-" * 30)
else:
    print("\n❌ No Dive Insight section found")

if article_data['article_paragraphs']:
    print(f"\n�� ARTICLE BODY ({len(article_data['article_paragraphs'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['article_paragraphs'][:3], 1):  # Show first 3
        print(f"\nParagraph {i}:")
        print(p[:200] + "..." if len(p) > 200 else p)
        print("-" * 30)
    if len(article_data['article_paragraphs']) > 3:
        print(f"... and {len(article_data['article_paragraphs']) - 3} more paragraphs")
else:
    print("\n❌ No regular article paragraphs found")

print("\n" + "="*80)

✅ Found 11 paragraphs using selector: article p

TITLE: Don't miss tomorrow's Manufacturing industry news

📅 Date: None
👤 Author: Unknown
�� Category: Unknown

🔍 DIVE INSIGHT SECTION (10 paragraphs):
--------------------------------------------------

Paragraph 1:
Isopropyl alcohol was the first chemical ExxonMobil produced over 100 years ago, according to the company. It is a byproduct of propylene processing , commonly found in hand sanitizers, household cleaning products, perfumes and lotions.
------------------------------

Paragraph 2:
In 1992, Exxon Mobil started increasing the purity of its cleaning solution for industrial uses, including for microchips, medical devices, laptops and gaming consoles.
------------------------------

Paragraph 3:
As microchips become even smaller, the company said it will need “ultra-pure” isopropyl alcohol to dry wafer surfaces, reduce impurities and prevent damage of the chips’ sensitive circuitry.
------------------------------

Paragraph 4:
To 

In [7]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"

# Add a user-agent + timeout for reliability
resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# --- Extract Title ---
title_tag = soup.find("h1")
title = title_tag.get_text(strip=True) if title_tag else "No title found"

# --- Extract Dive Insight Section (this is what you want!) ---
dive_insight_paragraphs = []
dive_insight_section = soup.find("h3", string="Dive Insight:")

if dive_insight_section:
    # Find the parent div that contains the Dive Insight section
    dive_insight_div = dive_insight_section.find_parent("div")
    if dive_insight_div:
        # Extract all paragraphs within this div
        dive_insight_nodes = dive_insight_div.find_all("p")
        dive_insight_paragraphs = [
            p.get_text(" ", strip=True)
            for p in dive_insight_nodes
            if p.get_text(strip=True)
        ]

# --- Extract Regular Article Paragraphs (fallback) ---
# Try multiple selectors since the structure might vary
article_paragraphs = []
selectors_to_try = [
    "div.article__body p",
    "div.article-content p", 
    "div.content p",
    "article p",
    "div[class*='article'] p",
    "div[class*='content'] p"
]

for selector in selectors_to_try:
    paragraph_nodes = soup.select(selector)
    if paragraph_nodes:
        article_paragraphs = [
            p.get_text(" ", strip=True)
            for p in paragraph_nodes
            if p.get_text(strip=True)
        ]
        print(f"✅ Found {len(article_paragraphs)} paragraphs using selector: {selector}")
        break

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag.get("datetime") if date_tag else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category (fallbacks included) ---
category_tag = soup.select_one(
    "a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link"
)
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "dive_insight": dive_insight_paragraphs,  # <-- This is the key section you want!
    "article_paragraphs": article_paragraphs,  # <-- Regular article content
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}

# --- Print Results ---
print("\n" + "="*80)
print(f"TITLE: {article_data['title']}")
print("="*80)

print(f"\n📅 Date: {article_data['meta']['date']}")
print(f"👤 Author: {article_data['meta']['author']}")
print(f"�� Category: {article_data['meta']['category']}")

if article_data['dive_insight']:
    print(f"\n🔍 DIVE INSIGHT SECTION ({len(article_data['dive_insight'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['dive_insight'], 1):
        print(f"\nParagraph {i}:")
        print(p)
        print("-" * 30)
else:
    print("\n❌ No Dive Insight section found")

if article_data['article_paragraphs']:
    print(f"\n�� ARTICLE BODY ({len(article_data['article_paragraphs'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['article_paragraphs'][:3], 1):  # Show first 3
        print(f"\nParagraph {i}:")
        print(p[:200] + "..." if len(p) > 200 else p)
        print("-" * 30)
    if len(article_data['article_paragraphs']) > 3:
        print(f"... and {len(article_data['article_paragraphs']) - 3} more paragraphs")
else:
    print("\n❌ No regular article paragraphs found")

print("\n" + "="*80)

✅ Found 11 paragraphs using selector: article p

TITLE: Don't miss tomorrow's Manufacturing industry news

📅 Date: None
👤 Author: Unknown
�� Category: Unknown

🔍 DIVE INSIGHT SECTION (10 paragraphs):
--------------------------------------------------

Paragraph 1:
Isopropyl alcohol was the first chemical ExxonMobil produced over 100 years ago, according to the company. It is a byproduct of propylene processing , commonly found in hand sanitizers, household cleaning products, perfumes and lotions.
------------------------------

Paragraph 2:
In 1992, Exxon Mobil started increasing the purity of its cleaning solution for industrial uses, including for microchips, medical devices, laptops and gaming consoles.
------------------------------

Paragraph 3:
As microchips become even smaller, the company said it will need “ultra-pure” isopropyl alcohol to dry wafer surfaces, reduce impurities and prevent damage of the chips’ sensitive circuitry.
------------------------------

Paragraph 4:
To 

In [2]:
import requests
from bs4 import BeautifulSoup
import json

url = "https://battery-news.de/en/2025/04/04/northvolt-lays-off-most-of-its-staff-in-sweden/"

resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# --- Extract Title ---
title_tag = soup.find("h1")
title = title_tag.get_text(strip=True) if title_tag else "No title found"

# --- Extract Regular Article Paragraphs ---
paragraphs = []
selectors_to_try = [
    "div.article__body p",
    "div.article-content p",
    "div.content p",
    "article p",
    "div[class*='article'] p",
    "div[class*='content'] p"
]

for selector in selectors_to_try:
    paragraph_nodes = soup.select(selector)
    if paragraph_nodes:
        raw_paragraphs = [
            p.get_text(" ", strip=True)
            for p in paragraph_nodes
            if p.get_text(strip=True)
        ]
        # 🔑 Convert into array of objects with p1, p2, ...
        paragraph_obj = {f"p{i+1}": text for i, text in enumerate(raw_paragraphs)}
        paragraphs = [paragraph_obj]  # put it inside a list
        break

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag.get("datetime") if date_tag else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category ---
category_tag = soup.select_one(
    "a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link"
)
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "paragraphs": paragraphs,  # now matches your example
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}

# ✅ Pretty-print JSON
print(json.dumps(article_data, indent=2, ensure_ascii=False))


{
  "title": "Northvolt Lays off Most of Its Staff in Sweden",
  "paragraphs": [
    {
      "p1": "According to several media reports, Swedish battery manufacturer Northvolt has laid off most of its employees as part of the ongoing insolvency proceedings. Approximately 1,700 employees will remain in Sweden. The company filed for insolvency in March 2025. At the time, Northvolt employed approximately 5,000 people. In November 2024, the battery manufacturer had already filed for Chapter 11 bankruptcy protection in the United States.",
      "p2": "The measures currently taken are intended to allow operations in Sweden to continue at a reduced level. According to the bankruptcy trustee, an agreement has been reached with key creditors and stakeholders. Specifically, ariund 1,200 of the original 3,000 jobs at the battery manufacturerʼs main plant in Skellefteå, Sweden, will be preserved. Foreign projects, such as the planned German battery production in Heide, will reportedly not be affec

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.manufacturingdive.com/news/exxonmobil-to-upgrade-baton-rouge-louisiana-plant-with-100-million-investment/744008/"

# Add a user-agent + timeout for reliability
resp = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# --- Extract Title ---
title_tag = soup.find("h1")
title = title_tag.get_text(strip=True) if title_tag else "No title found"

# --- Extract Dive Insight Section (this is what you want!) ---
dive_insight_paragraphs = []
dive_insight_section = soup.find("h3", string="Dive Insight:")

if dive_insight_section:
    # Find the parent div that contains the Dive Insight section
    dive_insight_div = dive_insight_section.find_parent("div")
    if dive_insight_div:
        # Extract all paragraphs within this div
        dive_insight_nodes = dive_insight_div.find_all("p")
        dive_insight_paragraphs = [
            p.get_text(" ", strip=True)
            for p in dive_insight_nodes
            if p.get_text(strip=True)
        ]

# --- Extract Regular Article Paragraphs (fallback) ---
# Try multiple selectors since the structure might vary
article_paragraphs = []
selectors_to_try = [
    "div.article__body p",
    "div.article-content p", 
    "div.content p",
    "article p",
    "div[class*='article'] p",
    "div[class*='content'] p"
]

for selector in selectors_to_try:
    paragraph_nodes = soup.select(selector)
    if paragraph_nodes:
        article_paragraphs = [
            p.get_text(" ", strip=True)
            for p in paragraph_nodes
            if p.get_text(strip=True)
        ]
        print(f"✅ Found {len(article_paragraphs)} paragraphs using selector: {selector}")
        break

# --- Extract Date ---
date_tag = soup.find("time")
date_utc = date_tag.get("datetime") if date_tag else None

# --- Extract Author ---
author_tag = soup.select_one("a.article-byline__author-link")
author = author_tag.get_text(strip=True) if author_tag else "Unknown"

# --- Extract Category (fallbacks included) ---
category_tag = soup.select_one(
    "a.topic-page-header__title-link, a.header-topic__title-link, a.breadcrumb__link"
)
category = category_tag.get_text(strip=True) if category_tag else "Unknown"

# --- Build Final Dict ---
article_data = {
    "title": title,
    "dive_insight": dive_insight_paragraphs,  # <-- This is the key section you want!
    "article_paragraphs": article_paragraphs,  # <-- Regular article content
    "meta": {
        "date": date_utc,
        "url": url,
        "category": category,
        "author": author,
    },
}

# --- Print Results ---
print("\n" + "="*80)
print(f"TITLE: {article_data['title']}")
print("="*80)

print(f"\n📅 Date: {article_data['meta']['date']}")
print(f"👤 Author: {article_data['meta']['author']}")
print(f"�� Category: {article_data['meta']['category']}")

if article_data['dive_insight']:
    print(f"\n🔍 DIVE INSIGHT SECTION ({len(article_data['dive_insight'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['dive_insight'], 1):
        print(f"\nParagraph {i}:")
        print(p)
        print("-" * 30)
else:
    print("\n❌ No Dive Insight section found")

if article_data['article_paragraphs']:
    print(f"\n�� ARTICLE BODY ({len(article_data['article_paragraphs'])} paragraphs):")
    print("-" * 50)
    for i, p in enumerate(article_data['article_paragraphs'][:3], 1):  # Show first 3
        print(f"\nParagraph {i}:")
        print(p[:200] + "..." if len(p) > 200 else p)
        print("-" * 30)
    if len(article_data['article_paragraphs']) > 3:
        print(f"... and {len(article_data['article_paragraphs']) - 3} more paragraphs")
else:
    print("\n❌ No regular article paragraphs found")

print("\n" + "="*80)

✅ Found 11 paragraphs using selector: article p

TITLE: Don't miss tomorrow's Manufacturing industry news

📅 Date: None
👤 Author: Unknown
�� Category: Unknown

🔍 DIVE INSIGHT SECTION (10 paragraphs):
--------------------------------------------------

Paragraph 1:
Isopropyl alcohol was the first chemical ExxonMobil produced over 100 years ago, according to the company. It is a byproduct of propylene processing , commonly found in hand sanitizers, household cleaning products, perfumes and lotions.
------------------------------

Paragraph 2:
In 1992, Exxon Mobil started increasing the purity of its cleaning solution for industrial uses, including for microchips, medical devices, laptops and gaming consoles.
------------------------------

Paragraph 3:
As microchips become even smaller, the company said it will need “ultra-pure” isopropyl alcohol to dry wafer surfaces, reduce impurities and prevent damage of the chips’ sensitive circuitry.
------------------------------

Paragraph 4:
To 

In [ ]:
{
  "title": "Northvolt Lays off Most of Its Staff in Sweden",
  "paragraphs": [
    {
      "p1": "According to several media reports, Swedish battery manufacturer Northvolt has laid off most of its employees as part of the ongoing insolvency proceedings. Approximately 1,700 employees will remain in Sweden. The company filed for insolvency in March 2025. At the time, Northvolt employed approximately 5,000 people. In November 2024, the battery manufacturer had already filed for Chapter 11 bankruptcy protection in the United States.",
      "p2": "The measures currently taken are intended to allow operations in Sweden to continue at a reduced level. According to the bankruptcy trustee, an agreement has been reached with key creditors and stakeholders. Specifically, ariund 1,200 of the original 3,000 jobs at the battery manufacturerʼs main plant in Skellefteå, Sweden, will be preserved. Foreign projects, such as the planned German battery production in Heide, will reportedly not be affected by the latest job cuts.",
      "p3": "Sources: https://news.bloomberglaw.com/bankruptcy-law/northvolt-cuts-workforce-to-1-700-as-part-of-bankruptcy-process https://www.reuters.com/business/autos-transportation/northvolt-bankruptcy-trustee-secures-agreement-scaled-down-operation-2025-03-31",
      "p4": "E-Mail-Adresse",
      "p5": "Vorname",
      "p6": "Nachname",
      "p7": "Ich habe die Datenschutzerklärung gelesen und stimme ihr zu.",
      "p8": "Battery-News © 2025 | Imprint | Privacy Policy"
    }
  ],
  "meta": {
    "date": null,
    "url": "https://battery-news.de/en/2025/04/04/northvolt-lays-off-most-of-its-staff-in-sweden/",
    "category": "Unknown",
    "author": "Unknown"
  }
}

In [9]:
import requests
from bs4 import BeautifulSoup
import html

def extract_chemxplore_article(url):
    """
    Extract article content from a ChemXplore article and format it similar to Manufacturing Dive structure
    """
    
    # Headers to mimic a real browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }
    
    try:
        # Make the request
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        # Parse the HTML
        soup = BeautifulSoup(response.text, "html.parser")
        
        # --- Extract Title ---
        title_tag = soup.find("title")
        title = title_tag.get_text(strip=True) if title_tag else "No title found"
        
        # --- Extract Date ---
        date_tag = soup.find("div", class_="article__date")
        date = date_tag.get_text(strip=True) if date_tag else None
        
        # --- Extract Paragraphs ---
        paragraphs = []
        
        # Find the article content area (look for common article content selectors)
        content_selectors = [
            "div.article__content",
            "div.article-content", 
            "div.content",
            "article",
            "div[class*='article']",
            "div[class*='content']"
        ]
        
        article_content = None
        for selector in content_selectors:
            article_content = soup.select_one(selector)
            if article_content:
                break
        
        if article_content:
            # Find all paragraphs and headings
            content_elements = article_content.find_all(["p", "h1", "h2", "h3", "h4", "h5", "h6"])
            
            for element in content_elements:
                text = element.get_text(strip=True)
                if text and len(text) > 10:  # Only add meaningful content
                    paragraphs.append(text)
        
        # If no structured content found, try to find paragraphs anywhere
        if not paragraphs:
            all_paragraphs = soup.find_all("p")
            for p in all_paragraphs:
                text = p.get_text(strip=True)
                if text and len(text) > 20:  # Filter out very short text
                    paragraphs.append(text)
        
        # --- Build the article data structure (similar to Manufacturing Dive) ---
        article_data = {
            "title": title,
            "paragraphs": paragraphs,
            "meta": {
                "date": date,
                "url": url,
                "category": "Unknown",  # ChemXplore might not have categories
                "author": "Unknown",     # ChemXplore might not have authors
                "source": "chemxplore"
            },
        }
        
        return article_data
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching article: {e}")
        return None
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        return None

def print_article_data(article_data):
    """Pretty print the extracted article data"""
    if not article_data:
        print("❌ No article data to display")
        return
    
    print("=" * 80)
    print(f"TITLE: {article_data['title']}")
    print("=" * 80)
    
    print(f"\n📅 Date: {article_data['meta']['date']}")
    print(f"👤 Author: {article_data['meta']['author']}")
    print(f"�� Category: {article_data['meta']['category']}")
    print(f"�� URL: {article_data['meta']['url']}")
    print(f"📰 Source: {article_data['meta']['source']}")
    
    if article_data['paragraphs']:
        print(f"\n�� ARTICLE PARAGRAPHS ({len(article_data['paragraphs'])} paragraphs):")
        print("-" * 50)
        for i, p in enumerate(article_data['paragraphs'], 1):
            print(f"\nParagraph {i}:")
            print(p)
            print("-" * 30)
    else:
        print("\n❌ No paragraphs found")

# Test with one of the ChemXplore article URLs
test_url = "https://chemxplore.com/news/lanxess-takes-action-to-counter-weak-market-environment"

print("�� Extracting ChemXplore article...")
print(f"URL: {test_url}\n")

try:
    article_data = extract_chemxplore_article(test_url)
    
    if article_data:
        print_article_data(article_data)
        
        # Show the JSON structure
        print("\n" + "=" * 80)
        print("JSON STRUCTURE:")
        print("=" * 80)
        import json
        print(json.dumps(article_data, indent=2, ensure_ascii=False))
        
    else:
        print("❌ Failed to extract article data")
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

�� Extracting ChemXplore article...
URL: https://chemxplore.com/news/lanxess-takes-action-to-counter-weak-market-environment

TITLE: LANXESS Responds to Market Challenges

📅 Date: 14 August 2025
👤 Author: Unknown
�� Category: Unknown
�� URL: https://chemxplore.com/news/lanxess-takes-action-to-counter-weak-market-environment
📰 Source: chemxplore

�� ARTICLE PARAGRAPHS (10 paragraphs):
--------------------------------------------------

Paragraph 1:
Financial Performance
------------------------------

Paragraph 2:
In Q2 2025, sales decreased by 12.6% to EUR 1.47 billion, influenced by portfolio and volume effects. EBITDA pre exceptionals fell 17.1% year-on-year to EUR 150 million. Despite challenging conditions, the company achieved a positive free cash flow of EUR 31 million and reduced net financial debt by 18% to EUR 2.069 billion.
------------------------------

Paragraph 3:
Guidance Adjustment
------------------------------

Paragraph 4:
Due to ongoing weak demand, the company adju